# Milan Traffic Forecasting 

This notebook adds everything
log-transformed SARIMA, clipped recursive
forecasts for LightGBM/LSTM, a documented hyperparameter-tuning log per
model, hardware capture for the timing table, sMAPE alongside MAPE, and it
saves every figure/table/log to `outputs/` 

**To use it:** drop all daily `.txt` files (All 62 files) into `DATA_DIR`, then run top to bottom. Nothing here assumes the
dataset is complete... the continuity gate in Section 2 will tell you clearly
if you're still missing something.


## 0. Setup

In [1]:
import glob, os, re, json, time, platform, itertools, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

DATA_DIR = 'data'
OUT_DIR = 'outputs'
LOG_DIR = os.path.join(OUT_DIR, 'logs')
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

EVAL_START = pd.Timestamp('2013-12-16 00:00')
EVAL_END = pd.Timestamp('2013-12-22 23:50')
FREQ_MINUTES = 10
SEASONAL_PERIOD = 144  # steps per day at 10-min resolution

pd.set_option('display.width', 120)


## 1. Hardware capture


In [2]:
def capture_hardware_info():
    info = {
        'python_version': platform.python_version(),
        'platform': platform.platform(),
        'processor': platform.processor() or platform.machine(),
        'cpu_count_logical': os.cpu_count(),
    }
    try:
        import psutil
        info['ram_total_gb'] = round(psutil.virtual_memory().total / 1e9, 1)
    except ImportError:
        info['ram_total_gb'] = 'psutil not installed -- run `pip install psutil` to capture this'

    gpu_info = 'no GPU backend detected'
    try:
        import torch
        if torch.cuda.is_available():
            gpu_info = torch.cuda.get_device_name(0)
    except ImportError:
        pass
    if gpu_info == 'no GPU backend detected':
        try:
            import tensorflow as tf
            gpus = tf.config.list_physical_devices('GPU')
            if gpus:
                gpu_info = str(gpus)
        except ImportError:
            pass
    info['gpu'] = gpu_info
    return info


HARDWARE_INFO = capture_hardware_info()
print(json.dumps(HARDWARE_INFO, indent=2))

with open(os.path.join(LOG_DIR, 'hardware_info.json'), 'w') as f:
    json.dump(HARDWARE_INFO, f, indent=2)
print(f"Saved -> {LOG_DIR}/hardware_info.json (paste this into the report's timing section)")


2026-09-30 10:27:03.208155: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


{
  "python_version": "3.9.6",
  "platform": "macOS-26.7-x86_64-i386-64bit",
  "processor": "i386",
  "cpu_count_logical": 16,
  "ram_total_gb": 68.7,
  "gpu": "no GPU backend detected"
}
Saved -> outputs/logs/hardware_info.json (paste this into the report's timing section)


## 2. Data loading & continuity gate

In [3]:
RAW_COLUMNS = ['square_id', 'time_interval', 'country_code',
               'sms_in', 'sms_out', 'call_in', 'call_out', 'internet']

def list_daily_files(data_dir):
    date_re = re.compile(r'(\d{4}-\d{2}-\d{2})')
    dated = []
    for p in sorted(glob.glob(os.path.join(data_dir, '*.txt'))):
        m = date_re.search(os.path.basename(p))
        if m:
            dated.append((pd.Timestamp(m.group(1)), p))
    dated.sort(key=lambda t: t[0])
    return dated


def report_missing_files(dated_files):
    if not dated_files:
        print('No files found in', DATA_DIR)
        return []
    start, end = dated_files[0][0], dated_files[-1][0]
    have = {d.normalize() for d, _ in dated_files}
    full_range = pd.date_range(start.normalize(), end.normalize(), freq='D')
    missing = [d for d in full_range if d not in have]
    print(f'Files found: {len(dated_files)} | span: {start.date()} -> {end.date()} '
          f'({len(full_range)} days) | missing files: {len(missing)}')
    if missing:
        print('  ', [d.date().isoformat() for d in missing])
    return missing


def load_raw_optimized(file_list, chunksize=2_000_000):
    use_cols = ['square_id', 'time_interval', 'internet']
    dtypes = {'square_id': 'int32', 'time_interval': 'int64', 'internet': 'float32'}
    parts = []
    for fp in file_list:
        chunk_sums = []
        for chunk in pd.read_csv(fp, sep='\t', header=None, names=RAW_COLUMNS,
                                  usecols=use_cols, dtype=dtypes, chunksize=chunksize):
            agg = chunk.groupby(['square_id', 'time_interval'], as_index=False)['internet'].sum()
            chunk_sums.append(agg)
        file_df = (pd.concat(chunk_sums, ignore_index=True)
                     .groupby(['square_id', 'time_interval'], as_index=False)['internet'].sum())
        parts.append(file_df)
    df = pd.concat(parts, ignore_index=True)
    df['square_id'] = df['square_id'].astype('int32')
    df['internet'] = df['internet'].astype('float32')
    mb = df.memory_usage(deep=True).sum() / 1e6
    return df, mb


def to_wide_datetime_index(df):
    out = df.copy()
    out['datetime'] = pd.to_datetime(out['time_interval'], unit='ms')
    wide = out.pivot_table(index='datetime', columns='square_id', values='internet', aggfunc='sum')
    return wide.sort_index()


def check_continuity(wide_df, freq_minutes=FREQ_MINUTES):
    full_index = pd.date_range(wide_df.index.min(), wide_df.index.max(), freq=f'{freq_minutes}min')
    have = set(wide_df.index)
    missing_steps = [t for t in full_index if t not in have]
    missing_days = sorted({t.normalize() for t in missing_steps})
    return {
        'expected_steps': len(full_index),
        'actual_steps': len(have & set(full_index)),
        'missing_steps': len(missing_steps),
        'missing_days': [d.date().isoformat() for d in missing_days],
        'is_continuous': len(missing_steps) == 0,
    }


def longest_contiguous_block(wide_df, freq_minutes=FREQ_MINUTES):
    idx = wide_df.index.sort_values()
    if len(idx) == 0:
        return None, None, 0
    step = pd.Timedelta(minutes=freq_minutes)
    gaps = (idx.to_series().diff() != step)
    gaps.iloc[0] = True
    block_id = gaps.cumsum()
    best_block = block_id.value_counts().idxmax()
    block_index = idx[block_id.values == best_block]
    return block_index.min(), block_index.max(), len(block_index)


def restrict_to_longest_block(wide_df, freq_minutes=FREQ_MINUTES):
    start, end, _ = longest_contiguous_block(wide_df, freq_minutes)
    return wide_df.loc[start:end]


dated_files = list_daily_files(DATA_DIR)
files = [p for _, p in dated_files]
missing_file_days = report_missing_files(dated_files)

full_df, full_mb = load_raw_optimized(files)
print(f'Loaded {len(files)} files -> {full_mb:.1f} MB, {len(full_df):,} rows')

wide = to_wide_datetime_index(full_df)
continuity = check_continuity(wide)
print(continuity)

wide_clean = restrict_to_longest_block(wide)
recheck = check_continuity(wide_clean)
assert recheck['is_continuous'], 'Restricted block is still not continuous.'
print('Clean working table:', wide_clean.shape, wide_clean.index.min(), '->', wide_clean.index.max())

eval_ok = (wide_clean.index.min() <= EVAL_START) and (EVAL_END <= wide_clean.index.max())
print('Eval window fully inside the gap-free block:', eval_ok)
assert eval_ok, ('STOP: Dec 16-22 is not fully covered by contiguous data yet. '
                  'Add the missing files listed above before continuing.')

with open(os.path.join(LOG_DIR, 'continuity_report.json'), 'w') as f:
    json.dump({'raw_continuity': continuity, 'missing_file_days':
               [d.date().isoformat() for d in missing_file_days]}, f, indent=2)
print(f'Saved -> {LOG_DIR}/continuity_report.json')


Files found: 62 | span: 2013-11-01 -> 2014-01-01 (62 days) | missing files: 0
Loaded 62 files -> 1427.9 MB, 89,245,318 rows
{'expected_steps': 8928, 'actual_steps': 8928, 'missing_steps': 0, 'missing_days': [], 'is_continuous': True}
Clean working table: (8928, 10000) 2013-10-31 23:00:00 -> 2014-01-01 22:50:00
Eval window fully inside the gap-free block: True
Saved -> outputs/logs/continuity_report.json


## 3. Metrics (MAE, MAPE, sMAPE, RMSE)
sMAPE added because MAPE blows up near-zero troughs...

In [4]:
def compute_metrics(actual, predicted, eps=1e-6):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    mae = np.mean(np.abs(actual - predicted))
    rmse = np.sqrt(np.mean((actual - predicted) ** 2))
    mape = np.mean(np.abs((actual - predicted) / np.clip(np.abs(actual), eps, None))) * 100
    smape = np.mean(2 * np.abs(actual - predicted) /
                     np.clip(np.abs(actual) + np.abs(predicted), eps, None)) * 100
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'sMAPE': smape}


## 4. Exploratory analysis
Same five outputs as before, now computed on `wide_clean` so the distribution
and decomposition reflect the full corrected dataset.


In [5]:
def top_n_areas(wide_df, n=3):
    totals = wide_df.sum(axis=0).sort_values(ascending=False)
    return list(totals.head(n).index), totals


top3, totals = top_n_areas(wide_clean, n=3)
print('Top 3 areas by total traffic:', top3)

fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(np.log1p(totals.values), bins=40)
ax.set_xlabel('log(1 + total internet traffic)')
ax.set_ylabel('Number of areas')
ax.set_title('Distribution of Total Internet Traffic Across Areas (completed data)')
fig.tight_layout()
fig.savefig(f'{OUT_DIR}/fig1_traffic_distribution.png', dpi=150)
plt.close(fig)


Top 3 areas by total traffic: [5161, 5059, 5259]


In [6]:
required_areas = top3 + [4159, 4556]
labels = [f'Area {a} (top-{i+1})' for i, a in enumerate(top3)] + ['Area 4159', 'Area 4556']

two_weeks_start = wide_clean.index.min()
two_weeks_end = two_weeks_start + pd.Timedelta(days=14)
window = wide_clean.loc[two_weeks_start:two_weeks_end, required_areas]

fig, ax = plt.subplots(figsize=(11, 5))
for area, label in zip(required_areas, labels):
    ax.plot(window.index, window[area], label=label, linewidth=1)
ax.set_xlabel('Date'); ax.set_ylabel('Internet traffic (activity units)')
ax.set_title('Internet Traffic -- First Two Weeks (completed data)')
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(f'{OUT_DIR}/fig2_first_two_weeks.png', dpi=150)
plt.close(fig)


In [7]:
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose

highest_area = top3[0]
series_highest = wide_clean[highest_area].dropna()

adf_stat, p_value = adfuller(series_highest)[:2]
adf_result = {'adf_statistic': adf_stat, 'p_value': p_value, 'stationary': p_value < 0.05}
print('ADF test:', adf_result)

lags = 200
acf_vals = acf(series_highest, nlags=lags)
pacf_vals = pacf(series_highest, nlags=lags)

fig, axes = plt.subplots(2, 1, figsize=(10, 7))
axes[0].stem(range(lags + 1), acf_vals); axes[0].axhline(0, color='red', lw=0.8)
axes[0].set_title(f'ACF -- Area {highest_area}')
axes[1].stem(range(lags + 1), pacf_vals); axes[1].axhline(0, color='red', lw=0.8)
axes[1].set_title(f'PACF -- Area {highest_area}')
fig.tight_layout()
fig.savefig(f'{OUT_DIR}/fig3_acf_pacf.png', dpi=150)
plt.close(fig)

with open(os.path.join(LOG_DIR, 'adf_test.json'), 'w') as f:
    json.dump(adf_result, f, indent=2)


ADF test: {'adf_statistic': -19.03189894699826, 'p_value': 0.0, 'stationary': True}


In [8]:
decomp = seasonal_decompose(series_highest, period=SEASONAL_PERIOD, model='additive',
                             extrapolate_trend='freq')

fig, axes = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
axes[0].plot(decomp.observed); axes[0].set_ylabel('Observed')
axes[1].plot(decomp.trend); axes[1].set_ylabel('Trend')
axes[2].plot(decomp.seasonal); axes[2].set_ylabel('Seasonal')
axes[3].plot(decomp.resid, '.', markersize=3); axes[3].set_ylabel('Resid')
fig.suptitle(f'Seasonal Decomposition -- Area {highest_area} (completed data)')
fig.tight_layout()
fig.savefig(f'{OUT_DIR}/fig4_decomposition.png', dpi=150)
plt.close(fig)

# Anomaly detection (>3 sigma on residuals) as a bonus diagnostic
resid = decomp.resid.dropna()
threshold = 3 * resid.std()
anomalies = resid[np.abs(resid) > threshold]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(series_highest.index, series_highest.values, linewidth=0.8)
ax.scatter(anomalies.index, series_highest.loc[anomalies.index], color='red', s=15,
           label='Anomaly (>3 sigma)', zorder=5)
ax.set_title(f'Anomaly Detection -- Area {highest_area}')
ax.legend()
fig.tight_layout()
fig.savefig(f'{OUT_DIR}/fig5_anomalies.png', dpi=150)
plt.close(fig)
print(f'{len(anomalies)} anomalous points flagged.')


138 anomalous points flagged.


## 5. Models
Each model now:
- fits on a **log1p-transformed** target (fixes SARIMA's negative-value problem
  and stabilizes variance for all three),
- inverse-transforms and **clips predictions at 0** before returning them,
- for recursive multi-step forecasting, **clips each fed-back prediction**
  to a sane historical range so errors can't compound into the runaway
  oscillation seen in the LightGBM blow-up.


In [9]:
from sklearn.metrics import mean_squared_error

def make_lag_features(series, n_lags=18, extra_lags=(144, 288)):
    """Lag features + hour-of-day / day-of-week cyclical features for LightGBM."""
    df = pd.DataFrame({'y': series})
    for lag in list(range(1, n_lags + 1)) + list(extra_lags):
        df[f'lag_{lag}'] = df['y'].shift(lag)
    df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)
    df['dow_sin'] = np.sin(2 * np.pi * df.index.dayofweek / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df.index.dayofweek / 7)
    return df.dropna()


def fourier_terms(index, period, K):
    """K harmonic sin/cos pairs for a given seasonal period, as exogenous
    regressors. Used instead of statsmodels' native `seasonal_order` term.
    """
    t = np.arange(len(index))
    cols = {}
    for k in range(1, K + 1):
        cols[f'sin_{period}_{k}'] = np.sin(2 * np.pi * k * t / period)
        cols[f'cos_{period}_{k}'] = np.cos(2 * np.pi * k * t / period)
    return pd.DataFrame(cols, index=index)


class SarimaModel:
    """ARIMA(order) with daily (and optionally weekly) seasonality supplied
    as Fourier-term exogenous regressors, rather than statsmodels' native
    `seasonal_order`.

    Why: native SARIMAX with seasonal_period=144 (daily seasonality at 10-min
    resolution) builds a state-space representation whose size scales with
    the seasonal period. On a multi-week training series this is extremely
    slow and memory-hungry -- in testing it was the direct cause of a kernel
    OOM crash during hyperparameter tuning. Fourier terms model the same
    daily/weekly cycles as plain regressors, so the state dimension only
    depends on `order`, not on the seasonal period. This is the standard
    approach recommended for high-frequency seasonal series (Hyndman &
    Athanasopoulos, *Forecasting: Principles and Practice*, ch. 12.1).
    """
    def __init__(self, order=(2, 0, 2), K_daily=3, K_weekly=0):
        self.order = order
        self.K_daily = K_daily
        self.K_weekly = K_weekly
        self.train_time_s = None
        self.predict_time_s = None

    def _exog(self, index):
        parts = [fourier_terms(index, SEASONAL_PERIOD, self.K_daily)]
        if self.K_weekly > 0:
            parts.append(fourier_terms(index, SEASONAL_PERIOD * 7, self.K_weekly))
        return pd.concat(parts, axis=1)

    def fit(self, series):
        from statsmodels.tsa.statespace.sarimax import SARIMAX
        self.train_index_ = series.index
        self.n_train_ = len(series)
        y_log = np.log1p(series.values)
        exog = self._exog(series.index)
        t0 = time.time()
        self.model_ = SARIMAX(y_log, order=self.order, exog=exog.values,
                               enforce_stationarity=False, enforce_invertibility=False)
        self.fit_ = self.model_.fit(disp=False)
        self.train_time_s = time.time() - t0
        return self

    def predict_n_steps(self, n_steps):
        future_index = pd.date_range(self.train_index_[-1] + pd.Timedelta(minutes=FREQ_MINUTES),
                                      periods=n_steps, freq=f'{FREQ_MINUTES}min')
        # Fourier terms must continue the same phase as training, so offset
        # the time index used inside _exog by the training length
        combined_index = self.train_index_.append(future_index)
        future_exog = self._exog(combined_index).iloc[self.n_train_:]
        t0 = time.time()
        forecast_log = self.fit_.forecast(steps=n_steps, exog=future_exog.values)
        self.predict_time_s = time.time() - t0
        return np.clip(np.expm1(forecast_log), 0, None)


class LightGBMModel:
    def __init__(self, n_lags=18, extra_lags=(144, 288), **lgbm_params):
        self.n_lags = n_lags
        self.extra_lags = extra_lags
        self.params = dict(n_estimators=300, num_leaves=31, learning_rate=0.05,
                            min_child_samples=20)
        self.params.update(lgbm_params)
        self.train_time_s = None
        self.predict_time_s = None

    def fit(self, series):
        import lightgbm as lgb
        y_log = np.log1p(series)
        feat_df = make_lag_features(y_log, self.n_lags, self.extra_lags)
        X, y = feat_df.drop(columns='y'), feat_df['y']
        t0 = time.time()
        self.model_ = lgb.LGBMRegressor(**self.params)
        self.model_.fit(X, y)
        self.train_time_s = time.time() - t0
        self.feature_cols_ = X.columns.tolist()
        return self

    def predict_recursive(self, train_series, n_steps, clip_quantile=0.999):
        """Recursive one-step forecasting with clipping to prevent the
        error-accumulation blow-up seen previously."""
        y_log_hist = list(np.log1p(train_series.values))
        idx_hist = list(train_series.index)
        cap = np.log1p(train_series.quantile(clip_quantile)) * 1.2  # soft ceiling

        preds = []
        t0 = time.time()
        cur_time = idx_hist[-1]
        for _ in range(n_steps):
            cur_time = cur_time + pd.Timedelta(minutes=FREQ_MINUTES)
            # build the feature row directly from the (small) tail of history --
            # do NOT reconstruct a Series over the whole growing history each
            # step, that turns this into an O(n^2) operation over long horizons
            row = {}
            for lag in list(range(1, self.n_lags + 1)) + list(self.extra_lags):
                row[f'lag_{lag}'] = y_log_hist[-lag]
            row['hour_sin'] = np.sin(2 * np.pi * cur_time.hour / 24)
            row['hour_cos'] = np.cos(2 * np.pi * cur_time.hour / 24)
            row['dow_sin'] = np.sin(2 * np.pi * cur_time.dayofweek / 7)
            row['dow_cos'] = np.cos(2 * np.pi * cur_time.dayofweek / 7)
            X_next = pd.DataFrame([row])[self.feature_cols_]

            y_next_log = self.model_.predict(X_next)[0]
            y_next_log = np.clip(y_next_log, 0, cap)  # <-- the fix: no runaway growth

            preds.append(y_next_log)
            y_log_hist.append(y_next_log)
            idx_hist.append(cur_time)

        self.predict_time_s = time.time() - t0
        return np.clip(np.expm1(np.array(preds)), 0, None)


class LstmModel:
    def __init__(self, lookback=144, units=32, epochs=15, batch_size=64):
        self.lookback = lookback
        self.units = units
        self.epochs = epochs
        self.batch_size = batch_size
        self.train_time_s = None
        self.predict_time_s = None

    def _windows(self, y_scaled):
        X, y = [], []
        for i in range(self.lookback, len(y_scaled)):
            X.append(y_scaled[i - self.lookback:i])
            y.append(y_scaled[i])
        return np.array(X)[..., np.newaxis], np.array(y)

    def fit(self, series):
        from sklearn.preprocessing import MinMaxScaler
        from tensorflow.keras.models import Sequential
        from tensorflow.keras.layers import LSTM, Dense
        from tensorflow.keras.callbacks import EarlyStopping

        y_log = np.log1p(series.values).reshape(-1, 1)
        self.scaler_ = MinMaxScaler()
        y_scaled = self.scaler_.fit_transform(y_log).ravel()

        X, y = self._windows(y_scaled)
        n_val = max(1, int(0.1 * len(X)))
        X_train, y_train = X[:-n_val], y[:-n_val]
        X_val, y_val = X[-n_val:], y[-n_val:]

        self.model_ = Sequential([
            LSTM(self.units, input_shape=(self.lookback, 1)),
            Dense(1),
        ])
        self.model_.compile(optimizer='adam', loss='mse')

        t0 = time.time()
        self.model_.fit(X_train, y_train, validation_data=(X_val, y_val),
                         epochs=self.epochs, batch_size=self.batch_size,
                         callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
                         verbose=0)
        self.train_time_s = time.time() - t0

        self.last_window_ = y_scaled[-self.lookback:].copy()
        return self

    def predict_recursive(self, train_series, n_steps, clip_quantile=0.999):
        window = self.last_window_.copy()
        cap_log = np.log1p(train_series.quantile(clip_quantile)) * 1.2
        cap_scaled = self.scaler_.transform([[cap_log]])[0, 0]

        preds_scaled = []
        t0 = time.time()
        for _ in range(n_steps):
            x = window[-self.lookback:].reshape(1, self.lookback, 1)
            y_next = self.model_.predict(x, verbose=0)[0, 0]
            y_next = np.clip(y_next, 0, cap_scaled)  # <-- prevents the flat/blow-up collapse
            preds_scaled.append(y_next)
            window = np.append(window, y_next)
        self.predict_time_s = time.time() - t0

        preds_log = self.scaler_.inverse_transform(np.array(preds_scaled).reshape(-1, 1)).ravel()
        return np.clip(np.expm1(preds_log), 0, None)


In [10]:
def _fit_lightgbm_with_local_fallback(self, series):
    y_log = np.log1p(series)
    feat_df = make_lag_features(y_log, self.n_lags, self.extra_lags)
    X, y = feat_df.drop(columns='y'), feat_df['y']
    t0 = time.time()

    try:
        import lightgbm as lgb
    except (ImportError, OSError) as exc:
        from sklearn.ensemble import HistGradientBoostingRegressor

        print(f'LightGBM unavailable ({type(exc).__name__}: {exc}); using sklearn fallback.')
        self.backend_ = 'sklearn HistGradientBoostingRegressor'
        self.model_ = HistGradientBoostingRegressor(
            max_iter=self.params.get('n_estimators', 300),
            max_leaf_nodes=self.params.get('num_leaves', 31),
            learning_rate=self.params.get('learning_rate', 0.05),
            min_samples_leaf=self.params.get('min_child_samples', 20),
            random_state=42,
        )
    else:
        self.backend_ = 'LightGBM'
        self.model_ = lgb.LGBMRegressor(**self.params)

    self.model_.fit(X, y)
    self.train_time_s = time.time() - t0
    self.feature_cols_ = X.columns.tolist()
    return self


LightGBMModel.fit = _fit_lightgbm_with_local_fallback

## 6. Hyperparameter tuning (documented, not just run once)
Each trial's parameters and validation performance are appended to
`tuning_log.csv` with a `notes` field for your reasoning 

Validation split: the **week before** the evaluation window (Dec 9-15) is
held out for tuning, so the true Dec 16-22 eval window stays untouched
until the final run.


In [11]:
VAL_START = EVAL_START - pd.Timedelta(days=7)
VAL_END = EVAL_START - pd.Timedelta(minutes=FREQ_MINUTES)

TUNING_AREA = top3[0]  # tune on the highest-traffic area, reuse best config everywhere
tune_series_full = wide_clean[TUNING_AREA].dropna()
tune_train = tune_series_full.loc[:VAL_START - pd.Timedelta(minutes=FREQ_MINUTES)]
tune_val_actual = tune_series_full.loc[VAL_START:VAL_END]
n_val_steps = len(tune_val_actual)

tuning_log_path = os.path.join(LOG_DIR, 'tuning_log.csv')
tuning_records = []

def log_trial(model_name, params, metrics, notes=''):
    tuning_records.append(dict(model=model_name, params=json.dumps(params),
                                notes=notes, timestamp=pd.Timestamp.now().isoformat(),
                                **metrics))
    pd.DataFrame(tuning_records).to_csv(tuning_log_path, index=False)


In [12]:
# --- SARIMA: small grid over (p,d,q) and number of daily Fourier harmonics ---
# (seasonality is handled via Fourier exogenous terms, not a native seasonal
# order -- see the SarimaModel docstring for why)
sarima_grid = [
    {'order': (1, 0, 1), 'K_daily': 2},
    {'order': (2, 0, 2), 'K_daily': 3},
    {'order': (2, 1, 2), 'K_daily': 3},
]
best_sarima_cfg, best_sarima_rmse = None, np.inf
for cfg in sarima_grid:
    m = SarimaModel(**cfg).fit(tune_train)
    pred = m.predict_n_steps(n_val_steps)
    metrics = compute_metrics(tune_val_actual.values, pred)
    log_trial('SARIMA', cfg, metrics,
              notes='grid search over (p,d,q) and number of daily Fourier harmonics, log-transformed series')
    print(cfg, metrics)
    if metrics['RMSE'] < best_sarima_rmse:
        best_sarima_rmse, best_sarima_cfg = metrics['RMSE'], cfg

print('Best SARIMA config:', best_sarima_cfg)


{'order': (1, 0, 1), 'K_daily': 2} {'MAE': 824.9134734974198, 'RMSE': 1223.3358105423554, 'MAPE': 44.00224117151209, 'sMAPE': 59.02035170969299}


/Users/Apple/Library/Python/3.9/lib/python/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


{'order': (2, 0, 2), 'K_daily': 3} {'MAE': 599.2067214024403, 'RMSE': 937.4628011954121, 'MAPE': 32.39713858286813, 'sMAPE': 40.47560837197358}
{'order': (2, 1, 2), 'K_daily': 3} {'MAE': 550.3860908334342, 'RMSE': 854.4785851563323, 'MAPE': 28.28067506177479, 'sMAPE': 34.31969967198723}
Best SARIMA config: {'order': (2, 1, 2), 'K_daily': 3}


In [13]:
# --- LightGBM: grid over depth/leaves and learning rate ---
lgbm_grid = [
    {'num_leaves': 15, 'learning_rate': 0.1, 'n_estimators': 200},
    {'num_leaves': 31, 'learning_rate': 0.05, 'n_estimators': 300},
    {'num_leaves': 63, 'learning_rate': 0.03, 'n_estimators': 500},
]
best_lgbm_cfg, best_lgbm_rmse = None, np.inf
for cfg in lgbm_grid:
    m = LightGBMModel(**cfg).fit(tune_train)
    pred = m.predict_recursive(tune_train, n_val_steps)
    metrics = compute_metrics(tune_val_actual.values, pred)
    log_trial('LightGBM', cfg, metrics,
              notes='grid search over num_leaves/learning_rate/n_estimators, recursive predictions clipped')
    print(cfg, metrics)
    if metrics['RMSE'] < best_lgbm_rmse:
        best_lgbm_rmse, best_lgbm_cfg = metrics['RMSE'], cfg

print('Best LightGBM config:', best_lgbm_cfg)


LightGBM unavailable (OSError: dlopen(/Users/Apple/Library/Python/3.9/lib/python/site-packages/lightgbm/lib/lib_lightgbm.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib
  Referenced from: <8FC36893-94B8-343C-9D9F-4CCBFE81B89B> /Users/Apple/Library/Python/3.9/lib/python/site-packages/lightgbm/lib/lib_lightgbm.dylib
  Reason: tried: '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/local/lib/libomp/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/local/lib/libomp/libomp.dylib' (no such file), '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '

In [14]:
# --- LSTM: grid over lookback window and hidden units ---
lstm_grid = [
    {'lookback': 72, 'units': 16, 'epochs': 15},
    {'lookback': 144, 'units': 32, 'epochs': 15},
    {'lookback': 288, 'units': 32, 'epochs': 20},
]
best_lstm_cfg, best_lstm_rmse = None, np.inf
for cfg in lstm_grid:
    m = LstmModel(**cfg).fit(tune_train)
    pred = m.predict_recursive(tune_train, n_val_steps)
    metrics = compute_metrics(tune_val_actual.values, pred)
    log_trial('LSTM', cfg, metrics,
              notes='grid search over lookback window and hidden units, early stopping patience=3, predictions clipped')
    print(cfg, metrics)
    if metrics['RMSE'] < best_lstm_rmse:
        best_lstm_rmse, best_lstm_cfg = metrics['RMSE'], cfg

print('Best LSTM config:', best_lstm_cfg)
print(f'\nFull tuning log saved -> {tuning_log_path}')


{'lookback': 72, 'units': 16, 'epochs': 15} {'MAE': 1152.9793111698984, 'RMSE': 1755.3592505740373, 'MAPE': 74.66091124349819, 'sMAPE': 79.29329276394088}
{'lookback': 144, 'units': 32, 'epochs': 15} {'MAE': 861.0537938682069, 'RMSE': 1331.8708303614205, 'MAPE': 50.036494298597724, 'sMAPE': 54.37144080403365}
{'lookback': 288, 'units': 32, 'epochs': 20} {'MAE': 1434.3022416146612, 'RMSE': 1895.7397752183301, 'MAPE': 161.88055955302477, 'sMAPE': 98.95679376585306}
Best LSTM config: {'lookback': 144, 'units': 32, 'epochs': 15}

Full tuning log saved -> outputs/logs/tuning_log.csv


## 7. Final experiments on the true evaluation window (Dec 16-22), all three target areas

In [15]:
TARGET_AREAS = {'highest_traffic': top3[0], 'square_4159': 4159, 'square_4556': 4556}
results_records = []
timing_records = []
all_predictions = {}

def run_all_models_for_area(area_id, area_label):
    s = wide_clean[area_id]
    train = s.loc[:EVAL_START - pd.Timedelta(minutes=FREQ_MINUTES)].dropna()
    eval_actual = s.loc[EVAL_START:EVAL_END].dropna()
    n_steps = len(eval_actual)

    preds = {}

    sarima = SarimaModel(**best_sarima_cfg).fit(train)
    preds['SARIMA'] = sarima.predict_n_steps(n_steps)
    timing_records.append(dict(area=area_label, model='SARIMA',
                                train_time_s=sarima.train_time_s,
                                predict_time_s=sarima.predict_time_s))

    lgbm = LightGBMModel(**best_lgbm_cfg).fit(train)
    preds['LightGBM'] = lgbm.predict_recursive(train, n_steps)
    timing_records.append(dict(area=area_label, model='LightGBM',
                                train_time_s=lgbm.train_time_s,
                                predict_time_s=lgbm.predict_time_s))

    lstm = LstmModel(**best_lstm_cfg).fit(train)
    preds['LSTM'] = lstm.predict_recursive(train, n_steps)
    timing_records.append(dict(area=area_label, model='LSTM',
                                train_time_s=lstm.train_time_s,
                                predict_time_s=lstm.predict_time_s))

    for model_name, pred_values in preds.items():
        m = compute_metrics(eval_actual.values, pred_values)
        results_records.append(dict(area=area_label, model=model_name, **m))

        fig, ax = plt.subplots(figsize=(11, 4))
        ax.plot(eval_actual.index, eval_actual.values, label='Actual')
        ax.plot(eval_actual.index, pred_values, '--', label='Predicted')
        ax.set_title(f'{model_name} -- {area_label} (Dec 16-22)')
        ax.set_ylabel('Internet traffic'); ax.legend()
        fig.autofmt_xdate(); fig.tight_layout()
        fig.savefig(f'{OUT_DIR}/forecast_{area_label}_{model_name}.png', dpi=150)
        plt.close(fig)

    return eval_actual, preds


for label, area_id in TARGET_AREAS.items():
    print(f'--- {label} (square {area_id}) ---')
    actual, preds = run_all_models_for_area(area_id, label)
    all_predictions[label] = (actual, preds)

results_df = pd.DataFrame(results_records)
timing_df = pd.DataFrame(timing_records)

for area_label in TARGET_AREAS:
    print(f'\n=== {area_label} ===')
    print(results_df[results_df.area == area_label]
          .set_index('model')[['MAE', 'MAPE', 'sMAPE', 'RMSE']])

results_df.to_csv(os.path.join(LOG_DIR, 'final_metrics.csv'), index=False)

timing_summary = timing_df.groupby('model')[['train_time_s', 'predict_time_s']].mean()
timing_summary['hardware'] = HARDWARE_INFO['processor']
timing_summary['gpu'] = HARDWARE_INFO['gpu']
timing_summary['method'] = 'averaged across the 3 target areas'
print('\nTiming (averaged across areas):')
print(timing_summary)
timing_summary.to_csv(os.path.join(LOG_DIR, 'timing_summary.csv'))

print(f'\nSaved -> {LOG_DIR}/final_metrics.csv, {LOG_DIR}/timing_summary.csv')


--- highest_traffic (square 5161) ---
LightGBM unavailable (OSError: dlopen(/Users/Apple/Library/Python/3.9/lib/python/site-packages/lightgbm/lib/lib_lightgbm.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib
  Referenced from: <8FC36893-94B8-343C-9D9F-4CCBFE81B89B> /Users/Apple/Library/Python/3.9/lib/python/site-packages/lightgbm/lib/lib_lightgbm.dylib
  Reason: tried: '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/local/lib/libomp/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/local/lib/libomp/libomp.dylib' (no such file), '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libo

## 8. Failure analysis
Worst-performing model on the highest-traffic area, with the top-20 worst errors highlighted.

In [16]:
actual_highest, preds_highest = all_predictions['highest_traffic']
worst_model = (results_df[results_df.area == 'highest_traffic']
               .sort_values('RMSE', ascending=False).iloc[0]['model'])

pred_series = pd.Series(preds_highest[worst_model], index=actual_highest.index)
errors = (actual_highest - pred_series).abs()
worst_idx = errors.sort_values(ascending=False).head(20).index

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(actual_highest.index, actual_highest.values, label='Actual')
ax.plot(actual_highest.index, pred_series.values, '--', label='Predicted')
ax.scatter(worst_idx, actual_highest.loc[worst_idx], color='red', s=25,
           label='Top-20 worst errors', zorder=5)
ax.set_title(f'Worst-performing periods -- {worst_model} on highest-traffic area')
ax.legend()
fig.autofmt_xdate(); fig.tight_layout()
fig.savefig(f'{OUT_DIR}/failure_analysis_{worst_model}.png', dpi=150)
plt.close(fig)

print(f'Worst-performing model on the highest-traffic area: {worst_model}')


Worst-performing model on the highest-traffic area: LSTM
